# Contextual Turn Embeddings — demo

A lightweight, CPU-friendly walkthrough:

1. build a toy dialogue dataset
2. convert it to the canonical format
3. get **base** turn embeddings (`f1`, with a mock fallback so it runs offline)
4. train a minimal **contextual** turn model (`f2`)
5. extract contextual embeddings
6. `save_pretrained` / `from_pretrained`
7. export `.npy` + `metadata.csv`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # run from a source checkout

import numpy as np
import pandas as pd
import torch

from contextual_turn_embeddings import (
    Config, ModelConfig, ContextualTurnModel,
    normalize_columns, encode_dialogues, export, train, set_seed,
)
set_seed(42)

## 1. A toy dialogue dataset

In [ ]:
rows = [
    ('d1', 0, 'hi, I need a hotel in Lujan', 'user'),
    ('d1', 1, 'sure, for how many nights?', 'system'),
    ('d1', 2, 'two nights please', 'user'),
    ('d1', 3, 'booked, anything else?', 'system'),
    ('d2', 0, 'book a table for two', 'user'),
    ('d2', 1, 'for what time?', 'system'),
    ('d2', 2, '8 pm tonight', 'user'),
    ('d3', 0, "what's the weather today?", 'user'),
    ('d3', 1, 'sunny and warm', 'system'),
]
df = pd.DataFrame(rows, columns=['dialogue_id', 'turn_id', 'utterance', 'speaker'])
df

## 2. Canonical format

`normalize_columns` validates the required columns, applies any column renames, and attaches a stable `row_id` used to keep exports aligned with the source rows.

In [ ]:
canonical = normalize_columns(df)
canonical.head()

## 3. Base embeddings (f1)

We try the real `BaseTurnEncoder` (needs `sentence-transformers`/`transformers`).
If unavailable, we fall back to deterministic **mock** embeddings so the rest of
the notebook still runs offline.

In [ ]:
try:
    from contextual_turn_embeddings import BaseTurnEncoder
    encoder = BaseTurnEncoder('sentence-transformers/all-MiniLM-L6-v2')
    base_embeddings = encoder.encode_texts(df['utterance'].tolist())
    print('Used BaseTurnEncoder, dim =', base_embeddings.shape[1])
except Exception as exc:
    print('Falling back to mock base embeddings:', type(exc).__name__)
    base_embeddings = np.random.default_rng(0).standard_normal((len(df), 32)).astype('float32')
    print('Mock base embeddings, dim =', base_embeddings.shape[1])

## 4. Train a minimal contextual model (f2)

Small dims + few epochs so it trains in seconds on CPU. `train()` infers the
input dimension from the base embeddings automatically.

In [ ]:
config = Config()
config.model = ModelConfig(
    input_dim=base_embeddings.shape[1],
    hidden_dim=32, num_heads=4, num_layers=2,
    max_turns=16, attention_mode='bidirectional',
    use_speaker_embeddings=True, num_speakers=4,
)
config.training.epochs = 5
config.training.batch_size = 4
config.training.output_dir = 'models/demo-contextual-turn'
config.data.max_turns = 16

model = train(config, df=df, embeddings=base_embeddings)

## 5. Extract contextual embeddings

In [ ]:
matrix, metadata = encode_dialogues(model, df, embeddings=base_embeddings)
print('contextual embeddings:', matrix.shape)
metadata.head()

## 6. Save and reload (Hugging Face-style)

In [ ]:
model.save_pretrained('models/demo-contextual-turn')
reloaded = ContextualTurnModel.from_pretrained('models/demo-contextual-turn')

matrix2, _ = encode_dialogues(reloaded, df, embeddings=base_embeddings)
print('reload matches:', np.allclose(matrix, matrix2, atol=1e-5))

## 7. Export `.npy` + `metadata.csv`

In [ ]:
export('outputs/demo_contextual_embeddings', matrix, metadata, config=config)
print(sorted(os.listdir('outputs/demo_contextual_embeddings')))

loaded = np.load('outputs/demo_contextual_embeddings/contextual_embeddings.npy')
meta = pd.read_csv('outputs/demo_contextual_embeddings/metadata.csv')
print('npy:', loaded.shape, '| metadata rows:', len(meta))
meta.head()